<a href="https://colab.research.google.com/github/raisharad/GenerativeAIandAgenticAI/blob/main/Prompting_and_sampling_algorithms_(upGrad).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Tutorial: LLM Prompting and APIs

In this tutorial, we will explore how to interact with Large Language Models (LLMs) using API-based inference. We will learn how different prompting strategies, generation parameters, and tool-calling capabilities influence a model's behavior, and how these features can be combined to build practical LLM-powered applications.

Throughout this notebook, we will:

- Send prompts to Large Language Models
- Design effective prompts using different prompting techniques
- Control model behavior using decoding parameters such as **temperature** and **top-p** sampling
- Build simple multi-turn chatbot interactions
- Enable LLMs to use external tools through **tool calling**
- Explore practical applications such as text summarization, code generation, text classification, logical reasoning etc.

The focus of this notebook is practical experimentation. Feel free to modify prompts, roles, and generation parameters throughout the exercises, and observe how these changes influence the model's outputs.

## 🛠️ Environment Setup

In this tutorial, we will use the OpenAI Python SDK to access large language models through a unified API interface.

### 📦 Install Required Packages

Run the following cell to install the OpenAI SDK:


In [1]:
!pip install -q --upgrade openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.6 MB/s eta 0:00:00


## 🔑 Storing API Keys in Google Colab Secrets

To avoid hardcoding API keys in notebooks:

1. Click the **🔑 Secrets** icon in the left sidebar of Google Colab.
2. Click **"Add a new secret"**.
3. Set the secret name to:

```
api_key
```

4. Paste your API key as the value.
5. Enable **Notebook Access** for the secret.

Once configured, the API key can be accessed securely within the notebook using:

In [ ]:
from google.colab import userdata

API_KEY = userdata.get("api_key")

SecretNotFoundError: Secret api_key does not exist.

### 🤗 Alternative: Using the Hugging Face Router

If you do not have access to an API key for the inference server used in this tutorial, you can instead access open-source language models through the **Hugging Face Router**.

To do this:

1. Create a Hugging Face account (if you do not already have one).
2. Navigate to **https://huggingface.co/settings/tokens**.
3. Generate a new access token.
4. Store the token in Google Colab Secrets using the name:

```text
hf_token
```

The token can then be accessed securely within the notebook using:




In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("hf_token")

## ⚙️ Global Configuration

Before running the experiments, we create a single OpenAI-compatible client that will be reused throughout the notebook.

The following cell allows you to choose between two different OpenAI-compatible inference backends:

- **Option 1:** An OpenAI-compatible inference server using an API key (`api_key`).
- **Option 2:** The Hugging Face Router using a Hugging Face access token (`hf_token`).

Simply uncomment the option you wish to use and comment out the other. The selected client and model will then be used for all subsequent examples in this notebook.




In [ ]:
from google.colab import userdata
from openai import OpenAI

# ------------------------------------------------------------------
# Choose ONE of the following methods.
# Uncomment the one you want to use.
# ------------------------------------------------------------------

# Option 1: API key for the OpenAI-compatible inference server
API_KEY = userdata.get("api_key")
BASE_URL = "http://103.42.50.41:443/api1"
MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"

# Option 2: Hugging Face Router
# API_KEY = userdata.get("hf_token")
# BASE_URL = "https://router.huggingface.co/v1"

# Choose any one model
# MODEL = "meta-llama/Llama-3.1-8B-Instruct"
# MODEL = "Qwen/Qwen2.5-7B-Instruct"
# MODEL = "Qwen/Qwen2.5-72B-Instruct"

# Create the client
client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

print(f"Using model: {MODEL}")

SecretNotFoundError: Secret api_key does not exist.

## 💬 Our First Prompt

Let's begin by sending a simple prompt to a language model.

In this example, we ask the model to briefly explain Transformers and observe the generated response.

This example also verifies that our API configuration is working correctly.


In [ ]:
# Send a prompt to the model
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Explain transformers briefly"
        }
    ],
    temperature=0.3,
    max_tokens=200
)
print(response.choices[0].message.content)

Transformers are a type of neural network architecture that has revolutionized the field of natural language processing (NLP). They were introduced in the paper "Attention Is All You Need" by Vaswani et al. in 2017. The key innovation of transformers is their use of self-attention mechanisms to weigh the importance of different words in a sentence when generating output.

Here’s a brief overview:

1. **Self-Attention Mechanism**: Transformers use self-attention to weigh the importance of each word in a sequence. This allows the model to focus on relevant parts of the input and ignore irrelevant ones. Each word in the sequence can be seen as attending to all other words, which helps the model capture long-range dependencies in text.

2. **Multi-Head Attention**: To improve the model's ability to capture various types of relationships between words, transformers use multiple attention heads. Each head focuses on different aspects of the data, and the results are combined to form a more c

### 📊 Understanding the API Response

In addition to the generated text, the API returns useful metadata about the request and response.

Some of the most commonly used fields include:

- **Model** – The model that generated the response.
- **Finish Reason** – Why the model stopped generating (e.g., **`stop`** for normal completion or **`length`** when the maximum token limit is reached).
- **Role** – The role associated with the generated message (typically **assistant**).
- **Tool Calls** – Indicates whether the model requested the use of an external tool or function.
- **Refusal** – Indicates whether the model refused to answer the request due to safety or policy constraints.
- **Token Usage** – Number of input and output tokens consumed.
- **Response ID** – A unique identifier for the API request.

Let's inspect these fields from the response object.

In [ ]:
print(response)

print("Response Metadata")
print("-" * 60)

print(f"Model              : {response.model}")
print(f"Response ID        : {response.id}")
print(f"Finish Reason      : {response.choices[0].finish_reason}")
print(f"Role               : {response.choices[0].message.role}")

print(f"Tool Calls         : {response.choices[0].message.tool_calls}")
print(f"Refusal            : {response.choices[0].message.refusal}")



print("\nToken Usage")
print("-" * 60)

print(f"Prompt Tokens      : {response.usage.prompt_tokens}")
print(f"Completion Tokens  : {response.usage.completion_tokens}")
print(f"Total Tokens       : {response.usage.total_tokens}")

ChatCompletion(id='chatcmpl-94f76cc09db03063', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='Transformers are a type of neural network architecture that has revolutionized the field of natural language processing (NLP). They were introduced in the paper "Attention Is All You Need" by Vaswani et al. in 2017. The key innovation of transformers is their use of self-attention mechanisms to process sequences of data, such as sentences or paragraphs.\n\nHere’s a brief overview:\n\n1. **Self-Attention Mechanism**: Instead of processing each word sequentially, transformers use self-attention to weigh the importance of different words in a sentence. This allows them to capture long-range dependencies and context more effectively.\n\n2. **Multi-Head Attention**: To improve the model\'s ability to handle different types of information, transformers use multiple attention heads. Each head focuses on different aspects of the input, and the re

## 🧠 Advanced Prompting



So far, our prompt consisted only of a user query. While this is often sufficient, modern chat-based language models allow us to provide additional instructions that influence the style, structure, and behavior of the response.

In the example below, we compare:

1. **A Basic Prompt** – only the user's question is provided.
2. **An Advanced Prompt** – the model is given a specific role and a set of instructions describing how it should answer.

In the given example, the advanced prompt asks the model to behave like an experienced Machine Learning instructor and provides additional guidance regarding:
- Explanation style
- Use of analogies
- Historical context
- Practical relevance

Run the following cell and compare the outputs generated by the basic and advanced prompts.

In [ ]:
USER_PROMPT = """
Why did transformers largely replace RNNs for modern NLP tasks?
"""

# BASIC PROMPT
basic_response = client.chat.completions.create(
    model=MODEL,
    temperature=0.7,
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": USER_PROMPT
        }
    ]
)

basic_output = basic_response.choices[0].message.content

# ADVANCED PROMPT
advanced_response = client.chat.completions.create(
    model=MODEL,
    temperature=0.7,
    max_tokens=300,
    messages=[
        {
            "role": "system",
            "content": """
You are an exceptional Machine Learning instructor.

When explaining a concept:
- Prioritize intuition over formal definitions.
- Use one simple analogy or mental model.
- Explain why the concept was important historically.
- Connect it to practical machine learning systems.
- Be concise but insightful.
"""
        },
        {
            "role": "user",
            "content": USER_PROMPT
        }
    ]
)

advanced_output = advanced_response.choices[0].message.content


print("\n" + "=" * 100)
print("BASIC PROMPT")
print("=" * 100)
print(basic_output)

print("\n\n" + "=" * 100)
print("ADVANCED PROMPT")
print("=" * 100)
print(advanced_output)


BASIC PROMPT
Transformers have largely replaced Recurrent Neural Networks (RNNs) in many Natural Language Processing (NLP) tasks due to several key advantages:

1. **Parallelization of Computation**: Unlike RNNs, which process sequences in a sequential manner (one token at a time), transformers can process all tokens in parallel. This is because the self-attention mechanism allows each token to be attended to by all other tokens simultaneously, leading to significant speedup in training and inference.

2. **Efficiency in Long Sequence Handling**: RNNs suffer from vanishing or exploding gradients when processing long sequences, as the error signal either diminishes or grows exponentially with the length of the sequence. Transformers mitigate this issue through their architecture, which does not rely on the sequential flow of information. The self-attention mechanism can handle long-range dependencies more effectively.

3. **Attention Mechanism**: The core innovation of transformers is 

## 🎲 Response Generation using Sampling Algorithms

So far, we have explored how changing the prompt can influence a model's response. However, the prompt is only one part of the generation process.

Language models assign probabilities to many possible next tokens. Decoding strategies determine how the next token is selected from this probability distribution.

Modern APIs expose several decoding parameters that allow us to control the balance between:
- Determinism and randomness
- Consistency and creativity
- Precision and diversity

### 🌡️ Temperature

Temperature controls the randomness of the generated response.

- Lower temperatures make the model more deterministic and focused on the most likely tokens.
- Higher temperatures increase randomness and creativity by allowing less likely tokens to be sampled.

To demonstrate this effect, we will ask the model to generate using different temperature values.

In [ ]:
PROMPT = """
Suggest one novel research direction using Large Language Models.

Keep the answer under 50 words.
"""

temperatures = [0.0, 1.0, 1.8]
num_runs = 2

for temp in temperatures:
    print("\n" + "=" * 100)
    print(f"TEMPERATURE = {temp}")
    print("=" * 100)

    for run in range(num_runs):
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": PROMPT}
            ],
            temperature=temp,
            max_tokens=300
        )

        print(f"\nRun {run + 1}:")
        print("-" * 50)
        print(response.choices[0].message.content)


TEMPERATURE = 0.0

Run 1:
--------------------------------------------------
Exploring LLMs for generative art creation and style transfer, integrating natural language descriptions with visual outputs.

Run 2:
--------------------------------------------------
Exploring LLMs for generative art creation and style transfer, integrating natural language descriptions with visual outputs.

TEMPERATURE = 1.0

Run 1:
--------------------------------------------------
Exploring LLMs for generative art creation and style transfer, integrating visual and textual understanding to produce novel artistic works.

Run 2:
--------------------------------------------------
Exploring generative models to create synthetic textual data for underrepresented languages, enhancing NLP model training diversity and global language support.

TEMPERATURE = 1.8

Run 1:
--------------------------------------------------
Explore LLMs for generative creativity in arts, combining text, images, and emotions to produc

### 🎯 Top-k and Top-p Sampling

In addition to **temperature**, modern language models often provide sampling strategies that further control how the next token is selected during generation.

#### Top-k Sampling

Top-k sampling restricts the model to selecting the next token from only the **k most probable candidates**.

- Smaller values of **k** make the model more conservative.
- Larger values of **k** allow more diverse generations.

Although Top-k is a commonly used decoding strategy, the OpenAI-compatible endpoints used in this tutorial do not expose a **top_k** parameter for the selected models.

#### Top-p (Nucleus) Sampling

Top-p sampling selects the next token from the **smallest set of tokens whose cumulative probability exceeds a threshold p**.

- Lower values of **p** produce more focused and deterministic outputs.
- Higher values of **p** allow the model to consider a larger set of candidate tokens, leading to more diverse generations.

In the following experiment, we generate **multiple completions** for the same prompt while varying the **Top-p** parameter. Compare the generated choices to observe how a lower **Top-p** tends to produce more focused and similar responses, whereas a higher **Top-p** generally results in greater diversity.

In [ ]:
PROMPT = """
Suggest one novel research direction using Large Language Models.

Keep the answer under 80 words.
"""

for p in [0.3, 1.0]:

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": PROMPT
            }
        ],
        temperature=0.8,
        top_p=p,
        n=3,                    # Generate 3 completions
        max_tokens=150
    )

    print("\n" + "=" * 100)
    print(f"TOP-P = {p}")
    print("=" * 100)

    for i, choice in enumerate(response.choices, start=1):
        print(f"\n--- Choice {i} ---\n")
        print(choice.message.content)


TOP-P = 0.3

--- Choice 1 ---

Explore using Large Language Models to generate and optimize complex scientific simulations and models across various disciplines, potentially accelerating discovery processes and enhancing predictive capabilities in fields like climate science, biology, and physics.

--- Choice 2 ---

Explore using Large Language Models to generate and optimize complex scientific simulations and models across various disciplines, potentially accelerating discovery processes and enhancing predictive capabilities in fields like climate science, bioinformatics, and materials science.

--- Choice 3 ---

Explore using Large Language Models to generate and optimize personalized learning paths for individual students based on their performance and interests, integrating real-time feedback and adaptive content delivery systems.

TOP-P = 1.0

--- Choice 1 ---

Exploring the use of Large Language Models (LLMs) to generate and optimize personalized learning paths in adaptive educa

## 👥 Roles in Prompting

Chat-based language models process conversations as a sequence of **messages**, where each message is assigned a specific role. These roles help the model distinguish between instructions, user queries, and previous responses, enabling more structured and context-aware interactions.

The three most commonly used roles are:

- **System**: Defines the overall behavior, personality, or constraints of the assistant.
- **User**: Represents the instructions or questions provided by the user.
- **Assistant**: Represents the model's previous responses, allowing the conversation to maintain context across multiple turns.

In the following example, we build a simple chatbot that maintains conversation history. As you interact with the chatbot, notice how previous user queries and assistant responses are preserved, allowing the model to answer follow-up questions while retaining context.

In [ ]:
conversation = [
    {
        "role": "system",
        "content": (
            "You are an experienced Machine Learning instructor. "
            "Provide clear, concise, and beginner-friendly explanations. "
            "Use bullet points where appropriate and include practical considerations."
        )
    }
]

print("Chatbot started! Type END to stop.\n")

while True:
    user_input = input("You: ")

    if user_input.strip() == "END":
        print("Ending chat...")
        break

    # Add user's message
    conversation.append(
        {
            "role": "user",
            "content": user_input
        }
    )

    # Get model response
    response = client.chat.completions.create(
        model=MODEL,
        messages=conversation,
        temperature=0.7,
        max_tokens=300
    )

    assistant_reply = response.choices[0].message.content

    # Print assistant response
    print(f"\nAssistant: {assistant_reply}\n")

    # Save assistant response for future context
    conversation.append(
        {
            "role": "assistant",
            "content": assistant_reply
        }
    )

Chatbot started! Type END to stop.

You: How do I LoRA fine-tune a model?

Assistant: LoRA (Low-Rank Adaptation) is a method to fine-tune large language models with minimal parameters. Here’s a step-by-step guide to get you started:

### Prerequisites
- Basic understanding of machine learning and deep learning.
- Python environment with necessary libraries installed.
- Access to a large language model (e.g., from Hugging Face).

### Step 1: Install Required Libraries
First, ensure you have the necessary libraries installed. You can use pip for this:

```bash
pip install transformers torch
```

### Step 2: Import Necessary Modules
Import the required modules in your Python script:

```python
from transformers import AutoModelForCausalLM, AutoTokenizer, LoraConfig, PeftModel
import torch
```

### Step 3: Load Pre-trained Model and Tokenizer
Load the pre-trained model and tokenizer:

```python
model_name = "bigscience/bloom"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = Au

## 🧰 Tool Calling

While Large Language Models are capable of reasoning about many tasks, they cannot directly execute code or perform external computations on their own.

**Tool calling** allows a language model to invoke external functions or APIs whenever a task requires specialized capabilities. Instead of attempting to solve the problem internally, the model can delegate the computation to an appropriate tool and use the returned result to generate its final response.

Typical applications of tool calling include:

- Mathematical and scientific computation
- Database queries
- Web search
- Weather and financial APIs
- File processing
- Code execution

In the following example, we compare the model's response with and without tool calling. The task is to compute the determinant of a large matrix—a computation that is best handled by a numerical library rather than by the language model itself.

In [ ]:
import json
import numpy as np

# Tool implementation
def compute_determinant(matrix):
    matrix = np.array(matrix)
    det = np.linalg.det(matrix)
    return str(det)


tools = [
    {
        "type": "function",
        "function": {
            "name": "compute_determinant",
            "description": "Compute the determinant of a square matrix.",
            "parameters": {
                "type": "object",
                "properties": {
                    "matrix": {
                        "type": "array",
                        "description": "A square matrix.",
                        "items": {
                            "type": "array",
                            "items": {
                                "type": "number"
                            }
                        }
                    }
                },
                "required": ["matrix"]
            }
        }
    }
]

# Generate a fixed random matrix
np.random.seed(42)
matrix = np.random.randint(-10, 11, size=(10, 10)).tolist()

PROMPT = f"""
Compute the determinant of the following matrix.

{matrix}
"""

# WITHOUT TOOL
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": PROMPT
        }
    ],
    temperature=0
)

print("=" * 100)
print("WITHOUT TOOL")
print("=" * 100)
print(response.choices[0].message.content)


# WITH TOOL
messages = [
    {
        "role": "user",
        "content": PROMPT
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto",
    temperature=0
)

tool_call = response.choices[0].message.tool_calls[0]

args = json.loads(tool_call.function.arguments)

determinant = compute_determinant(args["matrix"])

messages.append(response.choices[0].message)

messages.append(
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(determinant)
    }
)

final_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    temperature=0
)

print("\n\n" + "=" * 100)
print("WITH TOOL")
print("=" * 100)
print(final_response.choices[0].message.content)


# Ground Truth (Computed Directly)
true_determinant = compute_determinant(matrix)

print("\n\n" + "=" * 100)
print("GROUND TRUTH (NUMPY)")
print("=" * 100)
print(f"Determinant = {true_determinant}")

WITHOUT TOOL
To compute the determinant of a 10x10 matrix, we would typically use computational tools because the process is quite complex and time-consuming to do manually. However, I can guide you through the steps or use a computational tool to find the determinant for you.

Given the complexity, let's use a computational tool like Python with the NumPy library to calculate the determinant. Here’s how you can do it:

```python
import numpy as np

# Define the matrix
matrix = np.array([
    [-4, 9, 4, 0, -3, 10, -4, 8, 0, 0],
    [10, -7, -3, -8, 10, -9, 1, -5, -9, 10],
    [-10, 1, 1, 6, -1, 5, 4, 4, 8, 1],
    [9, -8, -6, 8, -4, 10, -2, -4, 7, -7],
    [3, 7, -2, 10, -9, 9, 4, -4, 1, -3],
    [4, -8, 3, 6, -7, 7, -3, -7, -9, -5],
    [-1, -7, 7, 1, -9, -1, -7, 3, 5, 4],
    [-3, 3, -3, 10, 5, 2, 7, 4, 10, 2],
    [-2, 4, 2, -10, -4, -2, -10, 1, -3, 0],
    [8, 6, -3, -8, -8, -10, -6, -1, -4, -2]
])

# Calculate the determinant
det = np.linalg.det(matrix)

print("The determinant of 

## 🔍 Understanding the Tool Calling Workflow

In the previous example, we saw how a language model can use an external tool to perform a computation. However, much of the interaction between the model and the tool happens behind the scenes.

To better understand the process, the following cell provides a **step-by-step view** of the complete tool-calling workflow. It prints the intermediate objects exchanged between the Python program and the language model, allowing us to observe how tool calling actually works.

In [ ]:
import json
import numpy as np
from pprint import pprint

# ============================================================
# TOOL IMPLEMENTATION
# ============================================================
def compute_determinant(matrix):
    print("\n[Python] Executing compute_determinant()")
    matrix = np.array(matrix)
    determinant = np.linalg.det(matrix)
    print(f"[Python] Determinant = {determinant}")
    return str(determinant)


# ============================================================
# TOOL DESCRIPTION
# ============================================================
tools = [
    {
        "type": "function",
        "function": {
            "name": "compute_determinant",
            "description": "Compute the determinant of a square matrix.",
            "parameters": {
                "type": "object",
                "properties": {
                    "matrix": {
                        "type": "array",
                        "description": "A square matrix.",
                        "items": {
                            "type": "array",
                            "items": {"type": "number"}
                        }
                    }
                },
                "required": ["matrix"]
            }
        }
    }
]

# ============================================================
# USER QUERY
# ============================================================
np.random.seed(42)
matrix = np.random.randint(-10, 11, (10, 10)).tolist()

messages = [
    {
        "role": "user",
        "content": f"Compute the determinant of the following matrix:\n\n{matrix}"
    }
]

print("=" * 100)
print("STEP 1: USER PROMPT")
print("=" * 100)
print(messages[0]["content"])

print("\n" + "=" * 100)
print("STEP 2: TOOLS AVAILABLE TO THE MODEL")
print("=" * 100)
pprint(tools)

# ============================================================
# FIRST MODEL CALL
# ============================================================
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto",
    temperature=0
)

assistant_message = response.choices[0].message

print("\n" + "=" * 100)
print("STEP 3: RAW ASSISTANT MESSAGE")
print("=" * 100)
pprint(assistant_message)

# ============================================================
# TOOL CALL
# ============================================================
tool_call = assistant_message.tool_calls[0]

print("\n" + "=" * 100)
print("STEP 4: TOOL CALL GENERATED BY THE MODEL")
print("=" * 100)

print(f"Function : {tool_call.function.name}")

print("\nArguments (JSON):")
print(tool_call.function.arguments)

args = json.loads(tool_call.function.arguments)

print("\nParsed Arguments:")
pprint(args)

# ============================================================
# EXECUTE TOOL
# ============================================================
print("\n" + "=" * 100)
print("STEP 5: PYTHON EXECUTES THE TOOL")
print("=" * 100)

tool_result = compute_determinant(**args)

print(f"\nReturned Value: {tool_result}")

# ============================================================
# SEND TOOL RESULT BACK
# ============================================================
messages.append(assistant_message)

tool_message = {
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(tool_result)
}

messages.append(tool_message)

print("\n" + "=" * 100)
print("STEP 6: CONVERSATION AFTER TOOL EXECUTION")
print("=" * 100)

print("\n----- USER MESSAGE -----")
pprint(messages[0])

print("\n----- ASSISTANT MESSAGE (Tool Call) -----")
pprint(assistant_message)

print("\n----- TOOL MESSAGE -----")
pprint(tool_message)

# ============================================================
# FINAL MODEL CALL
# ============================================================
final_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    temperature=0
)

print("\n" + "=" * 100)
print("STEP 7: FINAL ASSISTANT RESPONSE")
print("=" * 100)

print(final_response.choices[0].message.content)

STEP 1: USER PROMPT
Compute the determinant of the following matrix:

[[-4, 9, 4, 0, -3, 10, -4, 8, 0, 0], [10, -7, -3, -8, 10, -9, 1, -5, -9, 10], [-10, 1, 1, 6, -1, 5, 4, 4, 8, 1], [9, -8, -6, 8, -4, 10, -2, -4, 7, -7], [3, 7, -2, 10, -9, 9, 4, -4, 1, -3], [4, -8, 3, 6, -7, 7, -3, -7, -9, -5], [-1, -7, 7, 1, -9, -1, -7, 3, 5, 4], [-3, 3, -3, 10, 5, 2, 7, 4, 10, 2], [-2, 4, 2, -10, -4, -2, -10, 1, -3, 0], [8, 6, -3, -8, -8, -10, -6, -1, -4, -2]]

STEP 2: TOOLS AVAILABLE TO THE MODEL
[{'function': {'description': 'Compute the determinant of a square matrix.',
               'name': 'compute_determinant',
               'parameters': {'properties': {'matrix': {'description': 'A '
                                                                       'square '
                                                                       'matrix.',
                                                        'items': {'items': {'type': 'number'},
                                                      

## 🚀 Example Applications

So far, we have explored how to interact with Large Language Models using prompts, decoding parameters, roles, and tool calling.

In practice, the same API can be applied to a wide variety of Natural Language Processing (NLP) tasks simply by changing the prompt. In this section, we demonstrate a few common applications of LLMs, including:

- Text summarization
- Code generation
- Text classification
- Reasoning

Notice that the underlying model and API remain the same across all examples, the behavior changes only through the prompt provided to the model.

### 📝 Example 1: Text Summarization and Code Generation

Large Language Models are capable of understanding long pieces of text and performing multiple tasks in a single prompt.

In the following example, we provide a short legal clause and ask the model to:

1. Summarize the clause in a few bullet points.
2. Generate a Python function that implements the rule described in the clause.




In [ ]:
LEGAL_TEXT = """
A tenant must pay rent on or before the fifth day of each month.
If the rent is paid more than ten days after the due date without prior written approval from the landlord,
a late fee of $50 shall be charged.
"""

PROMPT = f"""
Read the following legal clause.

{LEGAL_TEXT}

Tasks:
1. Summarize the clause in at most two bullet points.
2. Generate a Python function

def calculate_late_fee(days_late):

that returns the appropriate late fee according to the clause.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": PROMPT
        }
    ],
    temperature=0.3,
    max_tokens=500
)

print(response.choices[0].message.content)

### Summary of the Clause

- Rent must be paid by the 5th of each month.
- A late fee of $50 is charged if rent is paid more than 10 days after the due date without prior written approval.

### Python Function

```python
def calculate_late_fee(days_late):
    """
    Calculate the late fee based on the number of days late.

    :param days_late: Number of days late the rent payment is.
    :return: Late fee amount as an integer.
    """
    if days_late > 10:
        return 50
    else:
        return 0
```

This function checks if the number of days late exceeds 10 and returns the late fee of $50 if it does, otherwise, it returns $0.


### 🏷️ Example 2: Text Classification

Many NLP tasks can be framed as classification problems. Here, we ask the model to determine whether a short piece of text is sarcastic and briefly justify its prediction.



In [ ]:
TEXT = """
Oh wonderful! Another software update that broke everything.
Exactly what I was hoping for today.
"""

PROMPT = f"""
Classify the following text as either:

- Sarcastic
- Not Sarcastic

Then explain your reasoning in one sentence.

Text:

{TEXT}
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": PROMPT
        }
    ],
    temperature=0.3,
    max_tokens=300
)

print(response.choices[0].message.content)

Sarcastic - The text expresses frustration through words that imply the opposite of the actual meaning, suggesting the person was not actually hoping for a broken software update.


### 🧩 Example 3: Reasoning

Beyond information retrieval and text generation, Large Language Models can also perform multi-step reasoning.

In the following example, the model is asked to solve a logic puzzle and explain its reasoning before presenting the final answer.

In [ ]:
PROMPT = """
Alice, Bob, Charlie, and Diana each own exactly one pet:
Dog, Cat, Rabbit, and Parrot.

Clues:

1. Charlie does not own the Dog or the Cat.
2. Bob owns neither the Cat nor the Rabbit.
3. Diana does not own the Parrot.
4. Alice does not own the Rabbit.
5. Charlie owns the Rabbit.
6. Bob does not own the Dog.
7. Alice does not own the Dog.

Determine who owns each pet.

Explain your reasoning before giving the final answer.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": PROMPT
        }
    ],
    temperature=0.3,
    max_tokens=500
)

print(response.choices[0].message.content)

Let's analyze the clues step by step to determine who owns each pet.

1. **Charlie does not own the Dog or the Cat.**
   - This means Charlie can only own the Rabbit or the Parrot.

2. **Bob owns neither the Cat nor the Rabbit.**
   - This means Bob can only own the Dog or the Parrot.

3. **Diana does not own the Parrot.**
   - This means Diana can only own the Dog, Cat, or Rabbit.

4. **Alice does not own the Rabbit.**
   - This means Alice can only own the Dog, Cat, or Parrot.

5. **Charlie owns the Rabbit.**
   - This directly tells us that Charlie owns the Rabbit.

6. **Bob does not own the Dog.**
   - This means Bob can only own the Parrot.

7. **Alice does not own the Dog.**
   - This means Alice can only own the Cat or the Parrot.

Now, let's summarize what we know:
- Charlie owns the Rabbit.
- Bob owns the Parrot (since he cannot own the Dog or the Rabbit, and Charlie already owns the Rabbit).
- Alice does not own the Rabbit or the Dog, so she must own the Cat.
- Diana does not

### 🎭 Example 4: Trick Question
In the following example, we present the model with a simple river-crossing puzzle. Although the puzzle appears similar to a well-known classic, it contains a subtle twist that tests whether the model carefully reasons about the given information rather than relying on familiar patterns.

In [ ]:
PROMPT = """
A farmer and a sheep are standing on one side of a river.
There is a boat with enough room for one human and one animal.
How can the farmer get across the river with the sheep in the fewest number of trips?
Give me the fewest number of trips.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": PROMPT
        }
    ],
    temperature=0.0,
    max_tokens=200
)

print(response.choices[0].message.content)

To solve this problem, we need to ensure that the sheep is not left alone with the farmer on either side of the river at any point, as that would imply the sheep could be eaten. Here's the step-by-step solution:

1. The farmer takes the sheep across the river.
2. The farmer returns alone to the original side.
3. The farmer takes the sheep back to the original side.
4. The farmer leaves the sheep on the other side and takes the boat across the river himself.

This sequence ensures that the sheep is never left alone with the farmer on either side of the river. Let's break it down:

1. Farmer + Sheep → Other Side (1 trip)
2. Farmer ← Original Side (1 trip)
3. Farmer + Sheep → Original Side (1 trip)
4. Farmer → Other Side (1 trip)

So, the total number of trips is 4.


### ⚠️ Example 5: Hallucination

Despite their impressive capabilities, Large Language Models can sometimes generate information that is **plausible-sounding but factually incorrect or entirely fabricated**. This phenomenon is commonly referred to as **hallucination**.

Hallucinations are more likely to occur when the model is asked about non-existent concepts, obscure topics, or information beyond its knowledge. Instead of admitting uncertainty, the model may confidently generate a convincing explanation.

**Please note that hallucination is not guaranteed.** Different models may respond differently, some may fabricate an answer, while others may explicitly state that they do not recognize the concept or do not have sufficient information.

In the following example, we ask the model about a fictitious machine learning algorithm and observe how it responds.

In [ ]:
PROMPT = """
Explain the Adaptive Quantum Adam Optimizer (AQAO) used in recent deep learning research.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": PROMPT
        }
    ],
    temperature=0.0,
    max_tokens=200
)

print(response.choices[0].message.content)

The Adaptive Quantum Adam Optimizer (AQAO) is a novel optimization algorithm that combines elements of quantum computing and traditional machine learning optimization techniques to improve the training process of deep neural networks. While the term "Adaptive Quantum Adam Optimizer" might not be a widely recognized term in the current literature, I can explain a conceptual framework that could encompass such an approach.

### Background

In deep learning, the Adam optimizer is a popular choice for gradient descent due to its adaptive learning rates and ability to handle sparse gradients. However, as models become more complex and datasets larger, there is a need for more sophisticated optimization methods that can further improve convergence and efficiency.

### Conceptual Framework of AQAO

1. **Quantum Computing Integration**:
   - **Quantum Gates**: Quantum gates can be used to perform operations on qubits, which can represent both 0 and 1 simultaneously (superposition). This allows